In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q --no-cache-dir google-search-results
import importlib, sys, pkgutil, subprocess, json

In [ ]:
import time
import pandas as pd
import os, requests
import geopandas as gpd
from shapely.geometry import Point

API_KEY = os.getenv("SERPAPI_API_KEY") or ""
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."

BASE = "https://serpapi.com/search.json"

def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

print("✅ SerpAPI Setup complete")


queries = ["shop","store"]


BASE_PARAMS = {
    "engine": "google_maps",
    "type": "search",
    "hl": "en",
    "gl": "uk",
    "google_domain": "google.co.uk",
    "ll": "@51.5246,-0.1340,12z"
}


CENTER_LAT = 51.5246
CENTER_LNG = -0.1340
DELTA_LAT = 0.06
DELTA_LNG = 0.09

LAT_MIN = CENTER_LAT - DELTA_LAT
LAT_MAX = CENTER_LAT + DELTA_LAT
LNG_MIN = CENTER_LNG - DELTA_LNG
LNG_MAX = CENTER_LNG + DELTA_LNG

MAX_RESULTS_TO_FETCH = 200
RESULTS_PER_PAGE = 20

all_rows = []
seen_place_ids = set()

for query in queries:
    print(f"\n🔍 Fetching: {query}")

    for start_index in range(0, MAX_RESULTS_TO_FETCH, RESULTS_PER_PAGE):
        print(f"  Start index: {start_index}")

        params = {
            **BASE_PARAMS,
            "q": query,
            "start": start_index,
            "num": RESULTS_PER_PAGE
        }

        try:
            res = serpapi_search("google_maps", params)
            results = res.get("local_results", []) or []

            if not results:
                print("  No more results.")
                break

            for r in results:
                pid = r.get("place_id")
                if pid in seen_place_ids:
                    continue
                seen_place_ids.add(pid)

                gps = r.get("gps_coordinates") or {}
                lat = gps.get("latitude")
                lng = gps.get("longitude")


                if lat and lng:
                    if not (LAT_MIN <= lat <= LAT_MAX and LNG_MIN <= lng <= LNG_MAX):
                        continue

                all_rows.append({
                    "query": query,
                    "title": r.get("title"),
                    "type": r.get("type"),
                    "rating": r.get("rating"),
                    "reviews": r.get("reviews"),
                    "address": r.get("address"),
                    "open_state": r.get("open_state"),
                    "place_id": pid,
                    "lat": lat,
                    "lng": lng,
                })

            if len(results) < RESULTS_PER_PAGE:
                break

            time.sleep(0.8)

        except Exception as e:
            print(f"⚠️ Error: {e}")
            break


df = pd.DataFrame(all_rows)
print(f"\n✅ Total unique places (filtered): {len(df)}")


df_points = df[['place_id', 'title', 'query', 'lat', 'lng']].dropna().drop_duplicates()
geometry = [Point(xy) for xy in zip(df_points['lng'], df_points['lat'])]
gdf = gpd.GeoDataFrame(df_points, geometry=geometry, crs="EPSG:4326")
gdf.to_file("shop_poi.geojson", driver="GeoJSON")
print("✅ GeoJSON exported: shop_poi.geojson")


df['has_reviews'] = df['reviews'].apply(lambda x: bool(x))
df['category'] = df['query']

csv_columns = [
    "query", "title", "type", "rating", "reviews", "address",
    "open_state", "place_id", "lat", "lng", "has_reviews", "category"
]

df.to_csv("shop_poi.csv", index=False, columns=csv_columns, encoding="utf-8-sig")
print("✅ CSV exported: shop_poi.csv")

In [ ]:
import pandas as pd
import time


df_paginated_places = df.copy()


all_reviews = []
seen_place_ids = set()

for _, row in df_paginated_places.iterrows():
    pid = row.get('place_id')
    title = row.get('title')
    query = row.get('query')

    if not pid or pid in seen_place_ids:
        continue

    seen_place_ids.add(pid)

    print(f"📍 Fetching reviews for: {title} ({query})")

    try:

        rev = serpapi_search('google_maps_reviews', {
            'place_id': pid,
            'hl': 'en',
            'sort_by': 'most_relevant'
        })

        reviews = rev.get('reviews', [])

        if not reviews:
            continue

        for r in reviews:
            all_reviews.append({
                'place_id': pid,
                'place_title': title,
                'category': query,
                'lat': row.get('lat'),
                'lng': row.get('lng'),
                'rating': r.get('rating'),
                'date': r.get('date'),
                'review_title': r.get('title'),
                'review_text': r.get('snippet'),
                'user_name': r.get('user', {}).get('name'),
                'local_guide': r.get('user', {}).get('local_guide'),
                'likes': r.get('likes'),
                'images': r.get('images'),
            })

        time.sleep(0.6)

    except Exception as e:
        print(f"⚠️ Error fetching {title}: {e}")
        continue

df_all_reviews = pd.DataFrame(all_reviews)
print(f"\n✅ Total reviews collected: {len(df_all_reviews)}")

# 3. Re-run merge logic (originally from deleted a12b89f4)
# df_paginated_places is already updated above.
# df_all_reviews is already updated above.

# Safety initializations for merge
df_results = df_all_reviews.copy() if 'df_all_reviews' in globals() else pd.DataFrame()

# Ensure key fields exist
if 'place_id' not in df_paginated_places.columns:
    raise ValueError("df_paginated_places must contain 'place_id'")

if df_results.empty:
    df_results = pd.DataFrame(columns=['place_id'])

if 'place_id' not in df_results.columns:
    raise ValueError("df_results must contain 'place_id' for merging")

# Deduplication
df_paginated_places = df_paginated_places.drop_duplicates(subset=['place_id'])
df_results = df_results.drop_duplicates(subset=['place_id', 'review_text', 'user_name'])

# Perform the merge
df_combined_data = pd.merge(
    df_paginated_places,
    df_results,
    on='place_id',
    how='left',
    suffixes=('_place', '_review')
)

# Optional optimizations
if 'query' not in df_combined_data.columns:
    print("⚠️ Warning: 'query' column missing")

df_combined_data['has_reviews'] = df_combined_data.filter(like='_review').notna().any(axis=1)
df_combined_data['category'] = df_combined_data['query']

print(f"✅ Total merged rows: {len(df_combined_data)}")

# 4. Save the combined data to a CSV file
df_combined_data.to_csv('/content/cafe reviews.csv', index=False)
print('✅ df_combined_data saved to /content/cafe reviews.csv')

# Display the head of the combined DataFrame
display(df_combined_data.head())